# Reconciliação: planilha nativa Omie (`centria_atualizado.xlsx` / aba `bdContas`) vs. API Omie

**Objetivo:** identificar quais movimentos financeiros trazidos pela API da Omie (`ListarMovimentos`) batem
com os lançamentos da exportação nativa (`bdContas`) — e, como subproduto, quais lançamentos existem só de
um lado (só na planilha, ou só na API) e por quê.

**Por que não é um simples merge por chave única:** a aba `bdContas` é o layout de exportação nativo da
Omie e **não traz `nCodTitulo`** (o identificador único do título na API) — só campos "de negócio"
(cliente, categoria, conta corrente, valor, datas, NC/Nfe). Então a reconciliação precisa ser heurística,
por múltiplas colunas em conjunto, com fallback progressivo quando a chave mais forte não está disponível
(ex.: título sem número de NF).

**Fonte da API:** em vez de reimplementar o enriquecimento (categorias/contas correntes/clientes), este
notebook reaproveita o pipeline já validado do projeto (`main_movimentos.py` → `src/report_builder.montar_geral`),
que monta a aba "Geral" **no mesmo layout e nomes de coluna da `bdContas`** — validado anteriormente contra
um relatório nativo real (ver docstring de `_dre_flag` em `src/report_builder.py`: bate em 99,96% dos
títulos). Isso elimina uma fonte inteira de divergência (diferenças de layout) e deixa a comparação focada
só no que interessa: quais lançamentos batem.

In [ ]:
import html
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

# Notebook mora em sondas/ — assume CWD = pasta do próprio notebook (padrão do
# Jupyter), então a raiz do repo (de onde "src" é importável) é o pai desta pasta.
sys.path.insert(0, str(Path.cwd().parent))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Layout padrão de exportação da Omie (aba "bdContas") — mesma ordem/nomes usados
# por `src/report_builder.py::_COLUNAS_GERAL`, para os dois lados usarem o schema idêntico.
COLUNAS_BDCONTAS = [
    "x", "Tipo", "Grupo", "Categoria", "Observação da Conta",
    "Data de Registro (completa)", "Data de Emissão (completa)", "NC/Nfe",
    "Data de Vencimento (completa)", "Situação do Vencimento", "Valor da Conta",
    "Pago ou Recebido", "A Pagar ou Receber", "Conta Corrente",
    "Cliente ou Fornecedor (Nome Fantasia)", "Observação do Pagto ou Recbto",
    "Data de Pagto", "COFINS Retido", "CSLL Retido", "INSS Retido", "IR Retido",
    "ISS Retido", "PIS Retido", "Desconto", "Juros", "DRE", "cod.fcx",
]

## 1. Carregar a planilha nativa (`bdContas`)

A aba tem duas linhas de "cabeçalho de exibição" acima do cabeçalho real (uma linha em branco e uma linha
com totais soltos em algumas colunas) — o cabeçalho de verdade está na 3ª linha física da planilha
(`header=2`, 0-indexed).

In [ ]:
df_native = pd.read_excel("../centria_atualizado.xlsx", sheet_name="bdContas", header=2)
df_native.columns = COLUNAS_BDCONTAS
df_native["_origem"] = "nativo"

print(f"{len(df_native)} lançamentos na planilha nativa")
df_native.head(3)

## 2. Carregar os dados da API — passo a passo

Em vez de chamar `main_movimentos.py` como caixa-preta e ler o `.xlsx` que ele produz, esta seção reproduz
aqui dentro cada etapa que o CLI (`src/cli_movimentos.py::main`) executa internamente — chamando exatamente
as mesmas funções do projeto (nenhuma lógica reimplementada, só exposta), pra deixar auditável de onde cada
dado vem:

1. `config.carregar_config()` — lê `OMIE_APP_KEY`/`OMIE_APP_SECRET` do `.env`.
2. `OmieClient(...)` — cliente HTTP com rate limit e retry.
3. `movimentos.buscar_movimentos(...)` — busca paginada em `financas/mf` (`ListarMovimentos`), sem filtro de
   período (todos os títulos já lançados na conta). Internamente já filtra por allow-list de `cGrupo`
   (mantém só `CONTA_A_PAGAR`/`CONTA_A_RECEBER`/`PREVISAO_CONTRATO`, descarta as baixas bancárias
   `CONTA_CORRENTE_PAG`/`REC` pra não duplicar valor) e adapta cada item pro shape
   `{"cabecTitulo": ..., "resumo": ...}` que o resto do pipeline espera.
4. `enrichment.build_categoria_map` / `build_conta_corrente_map` / `build_cliente_map` — cadastros de
   categorias, contas correntes e clientes/fornecedores, usados pra resolver descrições a partir dos
   códigos que vêm nos títulos. Lidos do cache local em `.cache/` (TTL 24h) quando disponível — só chamam a
   API se o cache estiver ausente ou vencido.
5. `report_builder.montar_geral(...)` — a mesma função que gera a aba "Geral" do relatório final, no layout
   padrão de exportação da Omie (`bdContas`). É o resultado desta função, montado ao vivo aqui, que vira
   `df_api` daqui pra frente.

In [4]:
from src.config import carregar_config
from src.omie_client import OmieClient
from src.movimentos import buscar_movimentos, contar_movimentos
from src.enrichment import build_categoria_map, build_conta_corrente_map, build_cliente_map
from src import report_builder

config = carregar_config()
client = OmieClient(
    app_key=config.app_key,
    app_secret=config.app_secret,
    max_req_por_segundo=config.max_req_por_segundo,
)
print("Cliente Omie configurado (chaves lidas de .env, sem exibi-las aqui).")

Cliente Omie configurado (chaves lidas de .env, sem exibi-las aqui).


### 2.1 Buscar os títulos (`financas/mf` / `ListarMovimentos`)

`contar_movimentos` faz uma sonda barata (1 registro) só pra saber o volume antes de paginar tudo —
mesma checagem que o CLI faz antes de avisar sobre buscas grandes. `buscar_movimentos` sem
`data_inicio`/`data_fim` traz todos os títulos já lançados na conta, sem filtro de período.

In [5]:
total_estimado = contar_movimentos(client)
print(f"Volume estimado antes de buscar: {total_estimado} movimentos (títulos + baixas + previsões de contrato)")

titulos_raw = buscar_movimentos(client)  # sem data_inicio/data_fim = todos os títulos já lançados
print(f"Títulos retornados após filtro de cGrupo e adaptação: {len(titulos_raw)}")
titulos_raw[0]

Volume estimado antes de buscar: 10791 movimentos (títulos + baixas + previsões de contrato)
Títulos retornados após filtro de cGrupo e adaptação: 5143


{'cabecTitulo': {'cCPFCNPJCliente': '04.962.772/0001-65',
  'cCodCateg': '1.01.99',
  'cGrupo': 'CONTA_A_RECEBER',
  'cHrAlt': '13:45:24',
  'cHrInc': '18:24:46',
  'cNatureza': 'R',
  'cNumDocFiscal': '00001471',
  'cNumParcela': '001/001',
  'cOrigem': 'MANR',
  'cRetCOFINS': 'S',
  'cRetCSLL': 'S',
  'cRetIR': 'S',
  'cRetPIS': 'S',
  'cStatus': 'ATRASADO',
  'cTipo': '99999',
  'cUsAlt': 'P000800884',
  'cUsInc': 'P000800884',
  'dDtAlt': '09/09/2024',
  'dDtEmissao': '02/05/2024',
  'dDtInc': '28/05/2024',
  'dDtPrevisao': '30/07/2024',
  'dDtRegistro': '28/05/2024',
  'dDtVenc': '30/07/2024',
  'nCodCC': 11061839888,
  'nCodCliente': 11066127939,
  'nCodTitRepet': 11066616943,
  'nCodTitulo': 11066616943,
  'nValorCOFINS': 1350,
  'nValorCSLL': 450,
  'nValorIR': 675,
  'nValorPIS': 292.5,
  'nValorTitulo': 35000},
 'resumo': {'cLiquidado': 'N',
  'nDesconto': 0,
  'nJuros': 0,
  'nMulta': 0,
  'nValAberto': 32232.5,
  'nValLiquido': 0,
  'nValPago': 0}}

### 2.2 Enriquecimento (categorias, contas correntes, clientes/fornecedores)

Cada `build_*_map` tenta o cache local em `.cache/` primeiro (TTL 24h); só bate na API se o cache estiver
ausente ou vencido — por isso, se você rodou `main.py`/`main_movimentos.py` nas últimas 24h, isto deve ser
quase instantâneo.

In [6]:
categoria_map = build_categoria_map(client)
cc_map = build_conta_corrente_map(client)
cliente_map = build_cliente_map(client)

print(f"{len(categoria_map)} categorias, {len(cc_map)} contas correntes, {len(cliente_map)} clientes/fornecedores")

140 categorias, 9 contas correntes, 357 clientes/fornecedores


### 2.3 Montar a aba "Geral" (layout `bdContas`)

`report_builder.montar_geral` é a mesma função usada por `cli_movimentos.py` pra gerar a aba "Geral" do
`.xlsx` final — já devolve as colunas nos mesmos nomes/ordem de `COLUNAS_BDCONTAS` (seção 1), então não
precisa renomear nada aqui.

In [7]:
df_api = report_builder.montar_geral(titulos_raw, categoria_map, cc_map, cliente_map)
assert list(df_api.columns) == COLUNAS_BDCONTAS, "montar_geral saiu de sincronia com COLUNAS_BDCONTAS"
df_api["_origem"] = "api"

print(f"{len(df_api)} lançamentos montados a partir da API")
df_api.head(3)

5143 lançamentos montados a partir da API


,x,Tipo,Grupo,Categoria,Observação da Conta,Data de Registro (completa),Data de Emissão (completa),NC/Nfe,Data de Vencimento (completa),Situação do Vencimento,Valor da Conta,Pago ou Recebido,A Pagar ou Receber,Conta Corrente,Cliente ou Fornecedor (Nome Fantasia),Observação do Pagto ou Recbto,Data de Pagto,COFINS Retido,CSLL Retido,INSS Retido,IR Retido,ISS Retido,PIS Retido,Desconto,Juros,DRE,cod.fcx,_origem
0,,2. Contas a Pagar,Despesas Diretas - Custo dos Serviços Prestados,Serviços de Terceiros Pessoa Jurídica (inativa),,2024-06-12,2024-06-12,NaN,2024-02-24,Pago,218.50,218.50,0.00,Itaú Unibanco,ENEL DISTRIBUICAO SAO PAULO,,2024-06-04,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,Sim,,api
1,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-09-20,2024-10-31,1478,2024-05-10,Vencido mais de 90 dias,21116.25,0.00,21116.25,Itaú Unibanco,MURTA,,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,Não,,api
2,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-09-24,2024-10-31,1473,2024-05-20,Recebido,15420.51,15420.51,0.00,Itaú Unibanco,KAINOS,,2024-09-30,0.0,0.0,0.0,0.0,0.0,0.0,15420.51,0.0,Sim,,api


## 3. Normalização

Duas diferenças sistemáticas entre as duas fontes, encontradas comparando os dados brutos antes de tentar
casar qualquer linha (nenhuma das duas é um erro de dado — são só convenções diferentes de exibição):

- **Sinal do valor**: na planilha nativa, "Valor da Conta" é negativo para Contas a Pagar; no relatório
  gerado pela API (`report_builder.montar_geral`) vem sempre positivo (o sinal não é aplicado nessa coluna).
  Solução: comparar por `abs(valor)`, usando a coluna "Tipo" (Pagar/Receber) para não misturar as duas
  naturezas.
- **Nome do cliente/fornecedor com HTML escapado**: alguns nomes têm `&` no cadastro (ex.: "GN&G
  CONSULTORIA"); a API devolve esse campo como `GN&amp;G CONSULTORIA` e `report_builder.montar_geral` não
  faz o unescape nessa coluna (só faz em "Observação da Conta", via `_observacao`) — então o mesmo
  cliente aparece com grafias diferentes nas duas fontes. Corrigido aqui com `html.unescape` antes de
  comparar. **Achado com potencial correção no código do projeto**: vale replicar esse unescape em
  `_COLUNAS_GERAL` / `montar_geral` (coluna "Cliente ou Fornecedor (Nome Fantasia)") em `src/report_builder.py`,
  para o relatório final também sair correto, não só esta comparação.

In [8]:
def _strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))


def _norm_text(v) -> str:
    """Texto pronto pra comparar: unescape de HTML, maiúsculas, sem acento, espaços colapsados."""
    if pd.isna(v):
        return ""
    s = html.unescape(str(v)).strip().upper()
    s = _strip_accents(s)
    return re.sub(r"\s+", " ", s)


def normalizar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["_valor_abs"] = df["Valor da Conta"].abs().round(2)
    df["_venc"] = pd.to_datetime(df["Data de Vencimento (completa)"], errors="coerce").dt.date
    df["_tipo"] = df["Tipo"].map(_norm_text)
    df["_cliente"] = df["Cliente ou Fornecedor (Nome Fantasia)"].map(_norm_text)
    df["_conta"] = df["Conta Corrente"].map(_norm_text)
    df["_categoria"] = df["Categoria"].map(_norm_text).str.replace(r"\s*\(INATIVA\)\s*$", "", regex=True)
    nc = df["NC/Nfe"].map(_norm_text)
    df["_nc"] = nc.where(~nc.isin(["", "N/D", "NAN", "NONE"]))
    df["_row_id"] = df.index
    return df


df_native = normalizar(df_native)
df_api = normalizar(df_api)

print("Nativo — Valor da Conta por Tipo:")
print(df_native.groupby("Tipo")["Valor da Conta"].agg(["min", "max"]))
print("\nAPI — Valor da Conta por Tipo:")
print(df_api.groupby("Tipo")["Valor da Conta"].agg(["min", "max"]))

Nativo — Valor da Conta por Tipo:
                            min         max
Tipo                                       
1. Contas a Receber        0.01  21893815.0
2. Contas a Pagar   -1106258.99        -1.0

API — Valor da Conta por Tipo:
                      min          max
Tipo                                  
1. Contas a Receber  0.01  75000000.00
2. Contas a Pagar    1.00   1210430.28


## 4. Reconciliação em fases

Sem `nCodTitulo` do lado nativo, a chave de casamento é heurística e vai enfraquecendo em fases — cada fase
só tenta casar o que sobrou da anterior (mesmo lançamento nunca é usado duas vezes):

1. **NC/Nfe + valor + vencimento** — quando os dois lados têm um número de NF preenchido, é a chave mais
   confiável.
2. **Tipo + valor + vencimento + conta corrente + cliente** — cobre a maioria dos lançamentos sem NF
   (débito automático, boleto sem nota, impostos, etc.).
3. **Tipo + valor + vencimento + conta corrente** (sem cliente) — rede de segurança para nomes de
   cliente/fornecedor divergentes entre as fontes.

Cada fase usa casamento por **multiconjunto** (não é um merge ingênuo): quando duas ou mais linhas têm a
mesma chave (ex.: parcelas mensais de mesmo valor), cada linha de um lado é pareada com uma única linha do
outro lado por posição dentro do grupo (`groupby(...).cumcount()`), então duplicatas não se multiplicam
nem "roubam" o pareamento de outra linha igual.

In [9]:
def casar_multiconjunto(nativo: pd.DataFrame, api: pd.DataFrame, chave: list[str]):
    """Casa `nativo` x `api` pela `chave`, tratando duplicatas de chave como um multiconjunto
    (cada linha repetida pareia com uma linha correspondente do outro lado, 1:1, sem explosão
    cartesiana). Retorna (pares_casados, sobra_nativo, sobra_api)."""
    n = nativo.copy()
    a = api.copy()
    n["_rank"] = n.groupby(chave).cumcount()
    a["_rank"] = a.groupby(chave).cumcount()
    pares = n.merge(a, on=chave + ["_rank"], suffixes=("_nativo", "_api"))

    ids_nativo = set(pares["_row_id_nativo"])
    ids_api = set(pares["_row_id_api"])
    return pares, nativo[~nativo["_row_id"].isin(ids_nativo)], api[~api["_row_id"].isin(ids_api)]


resultados_fase = []

# Fase 1 — NC/Nfe + valor + vencimento (só quando os dois lados têm NC/Nfe preenchido)
n_com_nc = df_native[df_native["_nc"].notna()]
a_com_nc = df_api[df_api["_nc"].notna()]
pares1, n1_sobra, a1_sobra = casar_multiconjunto(n_com_nc, a_com_nc, ["_tipo", "_nc", "_valor_abs", "_venc"])
resultados_fase.append(("1. NC/Nfe + valor + vencimento", pares1))

n_pool2 = pd.concat([n1_sobra, df_native[df_native["_nc"].isna()]])
a_pool2 = pd.concat([a1_sobra, df_api[df_api["_nc"].isna()]])

# Fase 2 — tipo + valor + vencimento + conta + cliente
pares2, n2_sobra, a2_sobra = casar_multiconjunto(n_pool2, a_pool2, ["_tipo", "_valor_abs", "_venc", "_conta", "_cliente"])
resultados_fase.append(("2. tipo + valor + vencimento + conta + cliente", pares2))

# Fase 3 — tipo + valor + vencimento + conta (sem cliente)
pares3, n3_sobra, a3_sobra = casar_multiconjunto(n2_sobra, a2_sobra, ["_tipo", "_valor_abs", "_venc", "_conta"])
resultados_fase.append(("3. tipo + valor + vencimento + conta", pares3))

nativo_sem_match = n3_sobra
api_sem_match = a3_sobra

for nome, pares in resultados_fase:
    print(f"Fase {nome}: {len(pares)} pares casados")

total_casado = sum(len(p) for _, p in resultados_fase)
print(f"\nTotal casado: {total_casado}")
print(f"Nativo sem correspondência na API: {len(nativo_sem_match)} de {len(df_native)} ({len(nativo_sem_match)/len(df_native):.1%})")
print(f"API sem correspondência na planilha: {len(api_sem_match)} de {len(df_api)} ({len(api_sem_match)/len(df_api):.1%})")

Fase 1. NC/Nfe + valor + vencimento: 1527 pares casados
Fase 2. tipo + valor + vencimento + conta + cliente: 3490 pares casados
Fase 3. tipo + valor + vencimento + conta: 0 pares casados

Total casado: 5017
Nativo sem correspondência na API: 166 de 5183 (3.2%)
API sem correspondência na planilha: 126 de 5143 (2.4%)


## 5. Resultado em R$ (não só em contagem de linhas)

Contagem de linhas trata um lançamento de R$ 5 igual a um de R$ 500 mil — a métrica que importa de verdade
pra reconciliação financeira é quanto do **valor total** da planilha nativa foi encontrado na API.

In [10]:
total_nativo = df_native["_valor_abs"].sum()
nao_encontrado = nativo_sem_match["_valor_abs"].sum()

print(f"Valor total na planilha nativa:        R$ {total_nativo:>15,.2f}")
print(f"Valor sem correspondência na API:      R$ {nao_encontrado:>15,.2f}  ({nao_encontrado/total_nativo:.2%})")
print(f"Valor com correspondência confirmada:  R$ {total_nativo - nao_encontrado:>15,.2f}  ({1 - nao_encontrado/total_nativo:.2%})")

print("\nPor Tipo:")
resumo_tipo = (
    df_native.groupby("Tipo")["_valor_abs"].sum().rename("total_nativo").to_frame()
    .join(nativo_sem_match.groupby("Tipo")["_valor_abs"].sum().rename("sem_match_nativo"))
    .fillna(0)
)
resumo_tipo["% batido"] = 1 - resumo_tipo["sem_match_nativo"] / resumo_tipo["total_nativo"]
resumo_tipo

Valor total na planilha nativa:        R$  129,311,197.26
Valor sem correspondência na API:      R$    1,372,457.20  (1.06%)
Valor com correspondência confirmada:  R$  127,938,740.06  (98.94%)

Por Tipo:


,total_nativo,sem_match_nativo,% batido
Tipo,,,
1. Contas a Receber,68316526.95,0.0,1.000000
2. Contas a Pagar,60994670.31,1372457.2,0.977499


## 6. O que não bateu, e por quê

Investigando os lançamentos sem correspondência (por conta corrente e categoria) dá pra ver que não é ruído
aleatório — são dois padrões concentrados e explicáveis:

In [11]:
print("=== Só na planilha nativa — por Conta Corrente ===")
print(nativo_sem_match["Conta Corrente"].value_counts())
print("\n=== Só na planilha nativa — por Categoria (top 10) ===")
print(nativo_sem_match["Categoria"].value_counts().head(10))

print("\n=== Só na API — por Conta Corrente ===")
print(api_sem_match["Conta Corrente"].value_counts())
print("\n=== Só na API — por Categoria (top 10) ===")
print(api_sem_match["Categoria"].value_counts().head(10))

=== Só na planilha nativa — por Conta Corrente ===
Conta Corrente
Caixinha         91
Itaú Unibanco    75
Name: count, dtype: int64

=== Só na planilha nativa — por Categoria (top 10) ===
Categoria
Serviços de Terceiros Pessoa Jurídica (inativa)          53
Material de Escritório                                   37
Despesas com Transporte                                  21
Lanches e Refeições                                      21
Saída de Transferência                                    6
IRPJ - Imposto de Renda Pessoa Juridica                   6
CSLL - Contribuicao Social                                5
IRRF Sobre Serviços Tomados                               3
CSRF Sobre PIS/COFINS/CSLL Retidos (Serviços Tomados)     3
Confraternização                                          2
Name: count, dtype: int64

=== Só na API — por Conta Corrente ===
Conta Corrente
Itaú Unibanco    104
Bradesco          21
Omie.CASH          1
Name: count, dtype: int64

=== Só na API — por Categoria (

**Padrão 1 — conta "Caixinha" quase sumiu da API.** A planilha nativa tem 94 lançamentos na conta
"Caixinha" (principalmente "Serviços de Terceiros PJ" e "Material de Escritório", pagos com dinheiro do
caixa pequeno); a API devolve só 3. Não é diferença de nome de conta (a string "Caixinha" existe idêntica
dos dois lados) nem de layout — os valores desses 91 lançamentos não aparecem em **nenhum** lugar da busca
da API (conferido buscando o valor exato, sem restringir por conta). Duas explicações prováveis, a
confirmar direto na Omie: (a) esses títulos foram excluídos/reclassificados para outra conta depois que a
planilha nativa foi exportada, ou (b) a conta "Caixinha" mudou de tipo/config na Omie de um jeito que a
tira do escopo de `ListarMovimentos` sem filtro de conta. Vale investigar puxando `ListarMovimentos` com
`cNatureza`/conta corrente explícitos pra essa conta especificamente.

**Padrão 2 — lançamentos "Retainer Fee" recentes só existem na API.** ~65 títulos a receber de categoria
"Retainer Fee" (contratos recorrentes) aparecem só na API, com NC/Nfe sequenciais (1507, 1508, 1547, ...) —
não existem na planilha nativa **nem com outro NC/Nfe** (conferido buscando o NC/Nfe direto na planilha).
Isso é consistente com a planilha nativa ser um **snapshot antigo**: à medida que os contratos recorrentes
seguem gerando parcelas novas na Omie, elas aparecem na API mas não retroagem pra um export já feito. Não é
uma divergência de dado — é a API estando "na frente" da planilha no tempo.

## 7. Entregável: movimentos da API que batem com a planilha nativa

Junta os pares casados das 3 fases num único DataFrame — as colunas originais da API (uma linha por
lançamento) mais a fase em que o casamento ocorreu (1 = mais confiável, 3 = mais heurística) e as colunas
correspondentes do lado nativo, pra auditoria. Grava também as sobras de cada lado em abas separadas no
mesmo arquivo, para revisão manual dos casos do item 6.

In [12]:
partes = []
for fase_num, (_, pares) in enumerate(resultados_fase, start=1):
    p = pares.copy()
    p["_fase"] = fase_num
    partes.append(p)

pares_todos = pd.concat(partes, ignore_index=True)

# Colunas do lado API no layout original bdContas (o que foi pedido: movimentos da API que batem),
# com um punhado de colunas do lado nativo do lado para conferência cruzada.
cols_api = [f"{c}_api" for c in COLUNAS_BDCONTAS]
cols_nativo_ref = [
    "NC/Nfe_nativo", "Data de Vencimento (completa)_nativo",
    "Cliente ou Fornecedor (Nome Fantasia)_nativo", "Valor da Conta_nativo",
]

movimentos_batidos = pares_todos[cols_api + cols_nativo_ref + ["_fase"]].copy()
movimentos_batidos.columns = COLUNAS_BDCONTAS + [f"{c} (nativo, conferência)" for c in [
    "NC/Nfe", "Data de Vencimento", "Cliente/Fornecedor", "Valor da Conta",
]] + ["Fase do casamento"]

print(f"{len(movimentos_batidos)} movimentos da API confirmados contra a planilha nativa")
movimentos_batidos.head(10)

5017 movimentos da API confirmados contra a planilha nativa


,x,Tipo,Grupo,Categoria,Observação da Conta,Data de Registro (completa),Data de Emissão (completa),NC/Nfe,Data de Vencimento (completa),Situação do Vencimento,Valor da Conta,Pago ou Recebido,A Pagar ou Receber,Conta Corrente,Cliente ou Fornecedor (Nome Fantasia),...,COFINS Retido,CSLL Retido,INSS Retido,IR Retido,ISS Retido,PIS Retido,Desconto,Juros,DRE,cod.fcx,"NC/Nfe (nativo, conferência)","Data de Vencimento (nativo, conferência)","Cliente/Fornecedor (nativo, conferência)","Valor da Conta (nativo, conferência)",Fase do casamento
0,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-05-28,2024-05-02,00001471,2024-07-30,Vencido mais de 90 dias,35000.00,0.00,32232.5,Bradesco,FISERV DO BRASIL INSTITUICAO DE PAGAMENTO LTDA,...,1350.00,450.00,0.0,675.00,0.0,292.50,0.00,0.0,Não,,00001471,2024-07-30,FISERV DO BRASIL INSTITUICAO DE PAGAMENTO LTDA,35000.00,1
1,,2. Contas a Pagar,Despesas Diretas - Custo dos Serviços Prestados,Adiantamento a Prestadores de Serviços,Pagamento Felipe,2024-06-05,2024-06-05,Inclusao Manual,2024-06-05,Pago,9800.00,9800.00,0.0,Itaú Unibanco,FELIPE GUIMARAES RODRIGUES DOS SANTOS,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,Sim,,Inclusao Manual,2024-06-05,FELIPE GUIMARAES RODRIGUES DOS SANTOS,-9800.00,1
2,,1. Contas a Receber,Receitas Diretas,Retainer Fee,Gerado automaticamente pela importação do extr...,2024-06-10,2024-06-10,1471,2024-06-10,Recebido,42232.50,42232.50,0.0,Bradesco,CIELO S.A - INSTITUICAO DE PAGAMENTO,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,Sim,,1471,2024-06-10,CIELO S.A - INSTITUICAO DE PAGAMENTO,42232.50,1
3,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-06-11,2024-06-11,1505,2024-06-25,Recebido,25000.00,23462.50,0.0,Itaú Unibanco,B2E,...,750.00,250.00,0.0,375.00,0.0,162.50,23462.50,0.0,Sim,,1505,2024-06-25,B2E,25000.00,1
4,,2. Contas a Pagar,Despesas Diretas - Custo dos Serviços Prestados,Serviços de Terceiros Pessoa Jurídica (inativa),,2024-06-11,2024-05-15,ND 1279,2024-06-11,Pago,951.51,951.51,0.0,Itaú Unibanco,HALABI SOCIEDADE DE ADVOGADOS,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,Sim,,ND 1279,2024-06-11,HALABI SOCIEDADE DE ADVOGADOS,-951.51,1
5,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-06-11,2024-06-11,1503,2024-06-25,Recebido,30000.00,28155.00,0.0,Bradesco,PODPAH PRODUCOES,...,900.00,300.00,0.0,450.00,0.0,195.00,0.00,0.0,Sim,,1503,2024-06-25,PODPAH PRODUCOES,30000.00,1
6,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-06-11,2024-06-11,1504,2024-06-22,Recebido,27367.25,25684.16,0.0,Itaú Unibanco,AG CAPITAL A CONSULTORIA E ASSESSORIA EMPRESAR...,...,821.02,273.67,0.0,410.51,0.0,177.89,25684.16,0.0,Sim,,1504,2024-06-22,AG CAPITAL A CONSULTORIA E ASSESSORIA EMPRESAR...,27367.25,1
7,,2. Contas a Pagar,Despesas Diretas - Custo dos Serviços Prestados,Serviços de Terceiros Pessoa Jurídica (inativa),Geovani,2024-06-11,2024-06-03,96,2024-06-11,Pago,4940.00,4940.00,0.0,Itaú Unibanco,GN&amp;G CONSULTORIA,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,Sim,,96,2024-06-11,GN&G CONSULTORIA,-4940.00,1
8,,2. Contas a Pagar,Despesas Diretas - Custo dos Serviços Prestados,Serviços de Terceiros Pessoa Jurídica (inativa),,2024-06-12,2024-06-12,FT-506P.2,2024-06-01,Pago,2529.75,2529.75,0.0,Bradesco,GREAT TRIPS VIAGENS E TURISMO,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,Sim,,FT-506P.2,2024-06-01,GREAT TRIPS VIAGENS E TURISMO,-2529.75,1
9,,1. Contas a Receber,Receitas Diretas,Retainer Fee,,2024-06-12,2024-06-12,1506,2024-06-20,Recebido,16330.97,15326.62,0.0,Bradesco,DINIE,...,489.93,163.31,0.0,244.96,0.0,106.15,0.00,0.0,Sim,,1506,2024-06-20,DINIE,16330.97,1


In [ ]:
caminho_saida = Path("../output/reconciliacao_bdcontas_vs_api.xlsx")
with pd.ExcelWriter(caminho_saida, engine="openpyxl") as writer:
    movimentos_batidos.to_excel(writer, sheet_name="Batidos (API)", index=False)
    nativo_sem_match[COLUNAS_BDCONTAS].to_excel(writer, sheet_name="So na planilha nativa", index=False)
    api_sem_match[COLUNAS_BDCONTAS].to_excel(writer, sheet_name="So na API", index=False)

print(f"Gravado em {caminho_saida.resolve()}")
print(f"  Batidos (API): {len(movimentos_batidos)}")
print(f"  Só na planilha nativa: {len(nativo_sem_match)}")
print(f"  Só na API: {len(api_sem_match)}")

## 8. Atualização: nova exportação nativa + filtro de cancelados

A exportação nativa foi refeita (`centria_atualizado.xlsx` — mesma aba `bdContas`, dados mais recentes que
os usados nas seções 1-7). Nessa rodada também investigamos por que ~193 títulos apareciam "só na API": a
maioria (81) tinha `cStatus="CANCELADO"` — e a planilha nativa **nunca** traz títulos cancelados (confirmado
direto: a coluna `Situação do Vencimento` não tem essa categoria em nenhuma das 5.183 linhas). Filtrando os
cancelados do lado da API antes de comparar, "só na API" caiu de 193 para **45** — o que sobrou (43 `PAGO`,
1 `A VENCER`, 1 `ATRASADO`, concentrado em despesas pequenas do dia a dia) é consistente com o mesmo "Padrão
2" já documentado na seção 6 (lançamentos que entraram na API depois do momento em que a planilha foi
exportada, não uma divergência de dado de verdade).

O resultado consolidado — títulos só na planilha nativa + títulos só na API, estes últimos com todos os
campos brutos do `cabecTitulo`/`resumo` (`nCodTitulo`, `nCodCliente`, `cStatus`, `cOrigem`, datas, valores e
retenções — o que ajuda a localizar cada um de volta na Omie) — está gravado em
`output/titulos_somente_um_lado.csv`. Célula abaixo abre esse arquivo.

In [ ]:
df_diff = pd.read_csv("../output/titulos_somente_um_lado.csv")
print(f"{len(df_diff)} linhas no total")
print(df_diff["_status"].value_counts())

print("\n=== Só na planilha nativa (amostra) ===")
display(df_diff[df_diff["_status"] == "somente_planilha_nativa"][
    ["Tipo", "Categoria", "Valor da Conta", "Data de Vencimento (completa)", "Cliente ou Fornecedor (Nome Fantasia)"]
].head(10))

print("\n=== Só na API (amostra, com campos brutos de identificação) ===")
display(df_diff[df_diff["_status"] == "somente_api"][
    ["Tipo", "Categoria", "Valor da Conta", "Data de Vencimento (completa)", "Cliente ou Fornecedor (Nome Fantasia)",
     "api_nCodTitulo", "api_nCodCliente", "api_cStatus", "api_cOrigem"]
].head(10))

## 9. Fechando os padrões da seção 6: rateio e Caixinha

A seção 6 identificou dois padrões nos lançamentos "só na planilha nativa" mas só especulou a causa. Uma
investigação separada (contra a conta real) fechou os dois:

**Padrão 1 (Caixinha) — causa confirmada, não é perda de dado.** `ListarExtrato` (`financas/extrato`) com
uma janela de datas larga devolve **vazio** para a conta "Caixinha" — não é a conta ter mudado de
tipo/escopo na Omie, é uma característica da API para esse volume/conta quando a janela é ampla. Pedindo
**mês a mês**, os 94 lançamentos aparecem normalmente. Essa descoberta está documentada como nota
permanente em `src/extrato.py` (perto de `buscar_extrato`), e o script que reproduz a recuperação mês a mês
está em `sondas/sonda_reconciliacao_bdcontas.py`.

**Achado adicional (rateio) — não é um padrão da seção 6, mas surgiu na mesma investigação.**
`ListarMovimentos` não expõe rateio de categoria por título (já documentado em `src/movimentos.py`), mas o
rateio **é** recuperável título a título via `ConsultarContaPagar` (`financas/contapagar`, campo
`categorias[]`) — confirmado contra títulos reais da RECEITA FAZENDA com mais de uma categoria. Não
integrado ao pipeline em lote (custo de uma chamada extra por título não se paga pro caso geral), mas
documentado em `src/movimentos.py` como caminho disponível se algum dia for necessário, e reproduzido em
`sondas/sonda_reconciliacao_bdcontas.py`.

Nenhuma das duas correções mudou o comportamento de `main.py`/`main_movimentos.py`/`main_dashboard.py` —
são achados documentados para uso futuro, não uma mudança no pipeline de produção.

## 10. Atualização — investigação aprofundada da Caixinha e do rateio (contra `ListarContasPagar`)

A seção 9 fechou os dois padrões da seção 6 com uma explicação qualitativa. Esta seção
registra a investigação **quantitativa** feita depois, cruzando os 166 registros "só na
nativa" direto contra `financas/contapagar · ListarContasPagar` (nunca usado antes neste
projeto) e `financas/extrato · ListarExtrato` com todos os campos. Resultado: **97% dos
166 (161) têm causa raiz confirmada**, dividido em três grupos bem distintos.

### 10.1 Conta "Caixinha" — previsão de Pedido de Compra, não título

A conta "Caixinha" tem **dois códigos cadastrados** (`11057330227` e `11564214808`) com
**94 lançamentos reais** (excluindo marcadores de saldo `cDesCliente="SALDO"`/
`"SALDO ANTERIOR"`). Testados exaustivamente:

| Endpoint | Resultado |
|---|---|
| `financas/mf · ListarMovimentos` | **0** — testado em janela larga e mês a mês, mesmo resultado |
| `financas/contacorrentelancamentos · ListarLancCC` | **0** — testado contra o histórico completo da empresa (5.530 lançamentos, todas as contas) |
| `financas/extrato · ListarExtrato` | **94/94 (100%)** — só aparecem aqui, e só recuperáveis buscando mês a mês (janela larga perde 58-61% dos registros) |

Um lançamento real (não-marcador) do extrato tem um conjunto de campos bem mais rico do
que o esperado — `cCodCategoria`, `cDesCategoria`, `cDocCliente` (CNPJ), `cNumero`,
`cDocumentoFiscal`, `cParcela`, `nCodLancamento` — mas **sempre** com
`cSituacao="Previsto"` + `cTipoDocumento="Pedido de Compra"` +
`cOrigem="Previsão de Pedido de Compra"`, e a categoria vem genérica
(`"0.01.02" "Saída de Transferência"`), não a categoria real da despesa.

**E depois, boa parte tem o pagamento de verdade registrado em outra conta, com atraso
(defasagem) de cerca de um mês.** Cruzando os 94 lançamentos contra `ListarContasPagar`
de Itaú Unibanco (3.467 títulos) + Bradesco (812 títulos), pareando 1:1 por valor + data
mais próxima (sem reaproveitar o mesmo título pra dois lançamentos da Caixinha):

| Tolerância de data | Pareados com título `PAGO` confirmado | Sem par |
|---|---:|---:|
| 90 dias | 56 de 94 (60%) | 38 |
| 180 dias | 57 de 94 (61%) | 37 |

Padrão por fornecedor: **GIMBA** (alimentação, recorrente) sempre acerta em
**Itaú Unibanco**, com defasagem de +27 a +46 dias (a previsão na Caixinha "atrasa" até
virar o pagamento real, tipicamente ~1 mês depois — em instâncias mais recentes essa
defasagem caiu pra -1 a -4 dias). **Editora Globo S.A.** (assinatura mensal fixa de
R$129,90) sempre acerta em **Bradesco**, com defasagem consistente de +25 a +31 dias.
Um caso (Amora Maker, R$8.000) teve defasagem invertida (-155 dias — o título real veio
*antes* da previsão).

**Os 37 sem par (39%)** concentram-se em **Kalunga SA** (papelaria — 9 ocorrências,
valores pequenos) e fornecedores de alimentação pontuais (Zona Cerealista, Garrama, Bold
Snacks, SK, Varanda Frutas, SPD Comércio), além de Evolução Ltda, FOF Suplementos e
Lenovo isolados. Hipótese não confirmada: pago por um caminho que `ListarContasPagar`
não cobre (cartão corporativo?).

Caminhos descartados: `ConsultarContaPagar(codigo_lancamento_omie=nCodLancamento)` —
"Lançamento não cadastrado" (é outro espaço de código); `produtos/pedidocompra ·
PesquisarPedCompra` pro período em torno de um lançamento específico — zero Pedidos de
Compra cadastrados, nem no ano inteiro.

### 10.2 Rateio de categoria — 93% do "Itaú Unibanco só-na-nativa" explicado

Dos 75 registros "só na nativa" da conta Itaú Unibanco (subconjunto diferente da
Caixinha), **70 (93%) são rateio**: um título único no `ListarMovimentos`/
`ListarContasPagar`, dividido em 2-3 linhas por categoria na planilha nativa. A
reconciliação por valor não bate com nenhuma parte porque cada parte só tem uma fração
do valor total do título.

- **20 registros — impostos ("RECEITA FAZENDA")**: 10 pares confirmados por NC/Nfe +
  data de vencimento idênticos (IRPJ+CSLL, PIS+COFINS, IRRF+CSRF, IRRF+INSS).
- **50 registros — despesa de viagem/reembolso a pessoa física**: 34 grupos
  "mesmo cliente + mesma data", categorias tipicamente "Despesas com Transporte" +
  "Lanches e Refeições" (às vezes + "Hospedagem"/"Confraternização") — formato clássico
  de relatório de despesas dividido por categoria no export nativo.

### 10.3 Dados sem nenhuma relação com a API

Depois de explicar Caixinha (94) e rateio (70), sobram **5 dos 166 registros originais**
sem nenhuma correspondência — testados contra `ListarMovimentos` (os 5 grupos de
`cGrupo`, dois campos de valor), `ListarLancCC` (histórico completo) e
`ListarContasPagar` (filtrado por Itaú, 3.467 títulos):

| Fornecedor | Valor | Vencimento (nativo) |
|---|---:|---|
| VERBENA FLORES | R$ 558,00 | 17/03/2026 |
| Editora Globo S.A. | R$ 129,90 | 20/04/2026 |
| Editora Globo S.A. | R$ 129,90 | 20/05/2026 |
| LC EMPREENDIMENTO IMOBILIARIO SPE LTDA | R$ 616,27 | 18/05/2026 |
| LOBBY TECNOLOGIA E PRODUTOS PERSONALIZADOS LTDA | R$ 177,25 | 07/08/2026 |

Somados aos 37 lançamentos da Caixinha sem par (seção 10.1), o **resíduo total sem
explicação recuperável é de 42 registros** — hipótese mais provável: título alterado ou
excluído na Omie depois do momento da exportação nativa, não confirmável sem `nCodTitulo`
do lado nativo.

> Nota: os registros da seção 10.3 são **distintos** dos lançamentos reais da Caixinha
> (seção 10.1), mesmo quando o nome do fornecedor coincide (ex.: "Editora Globo S.A." e
> "Lobby Tecnologia" aparecem nos dois grupos, com valores e datas diferentes em cada um)
> — são achados de investigações separadas que só compartilham fornecedor por
> coincidência comercial.

Documentação completa e reproduzível: `sondas/PESQUISA_RECONCILIACAO_CAIXINHA.md`.


## 11. Plano — mudanças para aproximar a base de dados da API da planilha nativa

Com as três causas da seção 10 mapeadas, este plano separa **o que é uma mudança de
código real** do que é **um limite que nenhuma mudança resolve** — pra não confundir
"achado explicado" com "achado corrigível". Nenhuma das mudanças abaixo foi
implementada; é um plano pra decisão futura, no mesmo espírito de
`PROPOSTAS_AUTOMACAO.md`.

### 11.1 Rateio de categoria (70 de 166 — o maior ganho possível, 42%)

**Mudança proposta:** trocar a fonte primária de títulos de `ListarMovimentos` para
`financas/contapagar · ListarContasPagar` + `financas/contareceber · ListarContasReceber`.
Diferente de `ConsultarContaPagar` (que exige uma chamada por título), esses dois métodos
de **listagem em lote** já devolvem o campo `categorias[]` corretamente preenchido pra
cada título, sem custo de chamada extra — confirmado na exploração (`categorias: [{
"codigo_categoria": "2.04.71", "percentual": 100, "valor": 1169 }]`).

**Por que resolve o rateio:** bastaria expandir 1 título em N linhas (uma por entrada de
`categorias[]`) na montagem do relatório — exatamente o que a planilha nativa já faz.

**O que se perde ao trocar a fonte:** `ListarContasPagar`/`ListarContasReceber` não
trazem `PREVISAO_CONTRATO` (previsão de faturamento de contrato ainda não lançada como
título) — isso só existe em `ListarMovimentos`. Solução: manter uma chamada
complementar a `ListarMovimentos`, filtrada só pro grupo `PREVISAO_CONTRATO` (47
registros no levantamento mais recente), sem rateio (não se aplica a previsão).

**Trabalho de implementação:**
1. Novo módulo de busca (equivalente a `movimentos.py`), paginando
   `ListarContasPagar`/`ListarContasReceber` — atenção: esses dois usam parâmetros em
   `snake_case` (`pagina`, `registros_por_pagina`, `filtrar_conta_corrente`...),
   diferente do `camelCase` de `ListarMovimentos` (`nPagina`, `nRegPorPagina`...).
2. Mapear os nomes de campo entre as fontes: `valor_documento`↔`nValorTitulo`,
   `status_titulo`↔`cStatus`, `data_vencimento`↔`dDtVenc`,
   `codigo_cliente_fornecedor`↔`nCodCliente`, etc.
3. Ajustar `report_builder.py::montar_geral` (e o equivalente na consulta `Geral` do
   Power Query) pra iterar `categorias[]` por título: cada entrada vira uma linha, com
   seu próprio `valor`/`codigo_categoria`, repetindo os demais campos do título (cliente,
   vencimento, conta corrente) — hoje a função assume 1 linha = 1 título = 1 categoria.
4. Juntar com a previsão de contrato (só grupo `PREVISAO_CONTRATO`) vinda de
   `ListarMovimentos`.
5. Rodar a reconciliação completa de novo (seções 1-10 deste notebook) contra a
   planilha nativa pra confirmar que a mudança não introduziu divergência nova.

**Custo/risco:** é uma mudança arquitetural real — reescreve a fonte de dado principal
da aba "Geral", não um ajuste pontual. Precisa de nova rodada de validação completa.

### 11.2 Conta "Caixinha" — previsões de Pedido de Compra (94 de 166, 57% já cobertos por título real)

**Recomendação: não replicar na aba "Geral".** Dois motivos:

- São dados de **natureza diferente** — previsão de orçamento interno
  (`cSituacao="Previsto"`, `cTipoDocumento="Pedido de Compra"`), não título formalizado.
  A planilha nativa mistura os dois conceitos numa mesma aba; replicar isso na API faria
  o relatório perder a distinção entre "título de verdade" e "previsão de caixa".
- **61% desses lançamentos já aparecem corretamente no relatório** — como o título real,
  formalizado ~1 mês depois em Itaú Unibanco ou Bradesco (seção 10.1). Trazer também a
  previsão da Caixinha **duplicaria esse valor**.

Só faria sentido considerar os **39% restantes (37 registros)** que não têm
contrapartida confirmada — mas a origem real desses ainda não foi identificada (cartão
corporativo? outro meio?), então não há hoje um endpoint certo pra buscar essa
informação. Antes de qualquer mudança de código aqui, o próximo passo seria investigar
como esses 37 são pagos de fato.

### 11.3 Os 5 órfãos + resíduo da Caixinha (42 de 166, 25%)

**Não há mudança de código que resolva isso.** É o limite inerente de qualquer
reconciliação automatizada comparando um snapshot (a planilha nativa, exportada num
momento fixo) contra uma API que reflete o estado atual — títulos podem ter sido
alterados/excluídos depois da exportação. Recomendação: aceitar como resíduo, resolvível
só por conferência manual pontual desses casos específicos direto na Omie, se algum dia
for necessário.

### 11.4 Resumo de impacto

| Mudança | Resolve quantos dos 166 | Esforço | Recomendação |
|---|---:|---|---|
| Trocar fonte pra `ListarContasPagar`/`ListarContasReceber` + expandir rateio | 70 (42%) | Alto — reescreve a montagem da aba "Geral" | **Fazer** — maior ganho, mudança bem definida |
| Incluir previsões da Caixinha na "Geral" | até 37 (22%), com risco de duplicar os outros 57 | Médio, resultado duvidoso | **Não fazer** — duplicaria valor pros 61% já cobertos por título real |
| Resolver os 5 órfãos + resíduo Caixinha | 0 — não automatizável | — | **Aceitar como limite** |

Ou seja: dos 166 originais, **97% (161) já está explicado**, mas só **42% (70)** é
de fato corrigível com uma mudança de código sem risco de piorar outra coisa — o resto
é uma diferença de conceito (Caixinha) ou um resíduo sem solução automatizável.
